# Phase 2 — 라이브러리 및 도구 개요

앞으로 사용할 도구를 직접 불러오고, 호출 방식과 색 순서 차이를 확인합니다. 각 셀을 순서대로 실행하십시오.

## 0. 그래프 한글 폰트 설정

matplotlib 는 기본 폰트에 한글이 없어 그래프 제목·축 라벨에 한글을 쓰면 깨질 수 있습니다. 아래 셀을 한 번 실행하면, 환경에 설치된 한글 폰트를 찾아 적용합니다. (한글 폰트가 없으면 그래프 라벨은 영어로 표기합니다.)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

candidates = ['Malgun Gothic', 'NanumGothic', 'AppleGothic',
              'Noto Sans CJK KR', 'Noto Sans KR']
available = {f.name for f in fm.fontManager.ttflist}
for name in candidates:
    if name in available:
        plt.rcParams['font.family'] = name
        print('한글 폰트 적용:', name)
        break
else:
    print('한글 폰트를 찾지 못했습니다 — 그래프 라벨은 영어로 표기합니다.')

plt.rcParams['axes.unicode_minus'] = False   # 음수 부호 깨짐 방지

## 1. import 와 별칭(as)
라이브러리는 `import` 로 불러오며, `as` 로 짧은 별칭을 붙입니다. 오류 없이 버전이 출력되면 정상입니다.
(hyperspy 는 Phase 4 에서 사용하며, 설치되어 있지 않으면 안내만 표시됩니다.)

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd

print('numpy     ', np.__version__)
print('opencv    ', cv2.__version__)
print('matplotlib', plt.matplotlib.__version__)
print('pandas    ', pd.__version__)

try:
    import hyperspy.api as hs
    print('hyperspy  ', hs.__version__)
except ModuleNotFoundError:
    print('hyperspy  : (Phase 4 에서 사용 예정 — 현재 환경에는 미설치)')

## 2. 라이브러리.함수() 호출
별칭 뒤에 점(`.`)을 붙여 기능을 호출합니다. 앞으로 등장하는 `np.~`, `cv2.~`, `plt.~` 는 모두 이 방식입니다.

In [ ]:
arr = np.zeros((3, 3))      # np 의 zeros 함수 호출
print(arr)

values = [12.1, 11.8, 12.4, 13.0]
print('평균 =', np.mean(values))   # Phase 1 의 average 함수를 한 줄로 대체

## 3. 도구 지도 요약

| 라이브러리 | 역할 | 사용 단계 |
|---|---|---|
| NumPy | 이미지를 배열로 다루는 토대 | Phase 3~ |
| OpenCV (cv2) | 이미지 처리·계측의 주력 엔진 | Phase 5~6 |
| Pillow (PIL) | 이미지 열기·저장·포맷 변환(보조) | Phase 4 |
| hyperspy | TEM TIFF 픽셀 scale 읽기 | Phase 4 |
| matplotlib | 이미지·그래프 시각화 | Phase 4·7 |
| pandas | 측정 데이터·통계 처리 | Phase 7 |

## 4. OpenCV(BGR) 와 Pillow·matplotlib(RGB) 의 색 순서

컬러 이미지는 각 픽셀이 **세 개의 숫자(채널)** 로 이루어집니다. 문제는 이 세 숫자의 **순서** 입니다.
- **RGB** : (Red, Green, Blue) 순서 — Pillow, matplotlib 이 사용
- **BGR** : (Blue, Green, Red) 순서 — OpenCV 가 사용

같은 픽셀값 `(255, 0, 0)` 이라도 RGB 로 해석하면 **빨강**, BGR 로 해석하면 **파랑**입니다. 데이터는 똑같은데 해석 순서가 다르면 색이 뒤바뀝니다. 아래에서 직접 확인합니다.

**(1) 같은 숫자, 다른 해석**

첫 픽셀이 `(255, 0, 0)` 인 이미지를 만들어, RGB 로 볼 때와 BGR 로 볼 때 색을 비교합니다.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

# (255, 0, 0) 으로 채운 작은 이미지
img = np.zeros((60, 60, 3), dtype=np.uint8)
img[:] = (255, 0, 0)

# 같은 배열의 채널 순서만 뒤집어 봄
img_swapped = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

fig, ax = plt.subplots(1, 2, figsize=(6, 3))
ax[0].imshow(img);         ax[0].set_title('(255,0,0) as RGB'); ax[0].axis('off')
ax[1].imshow(img_swapped); ax[1].set_title('(255,0,0) as BGR'); ax[1].axis('off')
plt.show()

print('원본 첫 픽셀 :', img[0, 0])           # [255   0   0]
print('순서 바꾼 뒤 :', img_swapped[0, 0])   # [  0   0 255]

왼쪽은 빨강, 오른쪽은 파랑으로 보입니다. **데이터는 동일하지만 채널 순서 해석만 바뀌어도 색이 완전히 달라진다**는 점을 알 수 있습니다.

**(2) 실제 상황: OpenCV 로 읽어 matplotlib 으로 표시할 때**

OpenCV(`cv2.imread`)로 읽은 컬러 이미지는 BGR 순서입니다. 이를 그대로 matplotlib(RGB 기준)으로 표시하면 색이 뒤집혀 보이므로, 표시 전에 `cv2.COLOR_BGR2RGB` 로 변환합니다.

In [ ]:
# 왼쪽 빨강 + 오른쪽 초록 이미지를 BGR 로 구성
#   (OpenCV 가 읽은 컬러 이미지가 이런 BGR 배열이라고 보면 됩니다)
bgr = np.zeros((60, 120, 3), dtype=np.uint8)
bgr[:, :60] = (0, 0, 255)   # BGR 의 (0,0,255) = 빨강
bgr[:, 60:] = (0, 255, 0)   # BGR 의 (0,255,0) = 초록

rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)   # 표시용 변환

fig, ax = plt.subplots(1, 2, figsize=(7, 3))
ax[0].imshow(bgr); ax[0].set_title('No convert (wrong)'); ax[0].axis('off')
ax[1].imshow(rgb); ax[1].set_title('BGR -> RGB (correct)'); ax[1].axis('off')
plt.show()

왼쪽(변환 안 함)은 의도한 빨강이 파랑으로 잘못 나오고, 오른쪽(변환 후)은 빨강·초록으로 정상 표시됩니다.

> **정리**: OpenCV 로 읽은 컬러 이미지를 matplotlib 으로 표시하거나 Pillow 와 함께 쓸 때는 `cv2.cvtColor(img, cv2.COLOR_BGR2RGB)` 로 순서를 맞춥니다. TEM 이미지는 대부분 흑백(단일 채널)이라 이 문제가 자주 발생하지는 않지만, 컬러를 다룰 때는 반드시 확인합니다.

---
## 직접 해보기
아래 `# TODO` 를 채우고 실행하십시오. `assert` 가 통과하면 `통과` 가 출력됩니다.

### 문제 1. import 와 버전 출력
`matplotlib.pyplot` 을 별칭 `plt` 로 불러오고, pandas 를 별칭 `pd` 로 불러오십시오. (이미 위에서 불러왔다면 다시 작성해도 무방합니다.)

In [ ]:
# TODO: import 두 줄 작성

assert 'plt' in dir() and 'pd' in dir()
print('통과')

### 문제 2. np 함수 호출
리스트 `vals = [10.0, 20.0, 30.0]` 의 평균을 NumPy 로 계산해 `m` 에 담으십시오.

In [ ]:
vals = [10.0, 20.0, 30.0]
m = # TODO  (np 사용)

assert m == 20.0
print('통과')

### 문제 3. 색 순서 변환
RGB 기준 '초록' `(0, 255, 0)` 배열을 만들고, OpenCV 로 BGR 로 변환한 결과를 `g_bgr` 에 담으십시오.
(초록은 변환 후에도 값 위치가 동일하지만, 변환 절차를 확인하는 것이 목적입니다.)

In [ ]:
g_rgb = np.zeros((1, 1, 3), dtype=np.uint8)
g_rgb[:] = (0, 255, 0)

g_bgr = # TODO  (cv2.cvtColor 사용)

assert list(g_bgr.flatten()) == [0, 255, 0]
print('통과')

---
이상으로 Phase 2 를 마칩니다. 다음 단계에서는 NumPy 로 이미지를 배열로 다룹니다.